# Etap IV: Modele Grawitacyjne Dostępności Przestrzennej i Interakcji WTC
    
Niniejszy notatnik odpowiada za implementację matematycznego algorytmu bilansującego czynniki podróży według paradygmatu modelu podwójnie ograniczonego (Doubly-Constrained Spatial Interaction Model Wilsona). 

Opiera się o koszty dojazdu wyliczone przez `r5py` w Etapie III ($c_{ij}$ interpretowane tu w ujęciu uogólnionym `cost_min_strict`), w zestawieniu wektorów rynkowych O/D. W celu sprowadzenia do homogenicznych wyników i uniknięcia nieporozumień z agregacjami różnych stref, silniki bilasujące wywoływane są *niezależnie dla poszczególnych aglomeracji*.

---
**Uwagi badawcze:**
* Dla Warszawy w użyciu znajduje się przygotowany skrypt `generate_warsaw_dummy_employment.py`, powołujący syntetyczną dystrybucję miejsc pracy oznaczonych w $1\text{km}\times1\text{km}$ docelowej przestrzeni badawczej, chroniąc silnik wyliczeniowy. 
* Jako miarę funkcji zniechęcenia do transportu (impedance decay) ufundowano tu logikę ekspotencjalną $f(c_{ij}) = e^{-\beta \cdot c_{ij}}$.
* **NOWOŚĆ METODOLOGICZNA:** Zgodnie z wytycznymi wprowadzono trójwariantową iterację modeli grawitacyjnych. Modele sztywne A i B ($\beta=0.03$, $\beta=0.15$) zestawiane są z wariantem C (wrażliwość ramy do siatki zamożności miasta `wealth_index` / `income_median`).

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import scipy.optimize
import json
from pathlib import Path
from datetime import datetime
import platform

# Ustalenie domyślnych ścieżek
ROOT = Path("..").resolve()
OUTPUTS = ROOT / "outputs"
INPUTS_E3 = OUTPUTS / "etap3"
OUTPUTS_E4 = OUTPUTS / "etap4"

OUTPUTS_E4.mkdir(parents=True, exist_ok=True)

# Definicja miast na wywołanie iteracyjne
CITIES = ["paris", "dublin", "warsaw"]

def _rel(p: Path) -> str:
    """Zwarca ścieżkę względnie do ROOT dla lżejszego indexu json."""
    try:
        return str(p.relative_to(ROOT))
    except ValueError:
        return str(p)

print(f"Środowisko gotowe: Python {platform.python_version()}")

Środowisko gotowe: Python 3.9.13


## Konstrukcja Silnika Modeu: Algorytm IPF (Doubly-Constrained)

In [2]:
def doubly_constrained_gravity(cost_matrix_df, O_col='O_pop', D_col='D_jobs', beta=0.08, tol=1e-5, max_iter=200):
    """
    Implementuje proceduralny algorytm Wilsona oparty na Iterative Proportional Fitting (IPF).
    Obsługuje również dynamiczne przypisywanie parametru beta z kolumny.
    """
    df = cost_matrix_df.copy()
    
    # 1. Funkcja impedancji
    if isinstance(beta, str) and beta in df.columns:
        df['f_cij'] = np.exp(-df[beta] * df['cost'])
    else:
        df['f_cij'] = np.exp(-beta * df['cost'])
    
    # Wyrównanie marginaliów (kategoryczny warunek stabilności IPF dla modeli Double-Constrained)
    # Zabezpiecza algorytm przed rozbieżnością do nieskończoności
    O_target = df.groupby('origin_id')[O_col].first()
    D_target = df.groupby('dest_id')[D_col].first()
    
    total_O = O_target.sum()
    total_D = D_target.sum()
    
    if total_D == 0 or total_O == 0:
        raise ValueError("Suma popytu lub podaży wynosi 0! Bilansowanie niemożliwe.")
        
    scale_factor = total_O / total_D
    D_target_scaled = D_target * scale_factor
    
    df['O_i'] = df['origin_id'].map(O_target).fillna(0)
    df['D_j'] = df['dest_id'].map(D_target_scaled).fillna(0)
    
    # 2. Inicjalizacja czynników
    df['A_i'] = 1.0
    df['B_j'] = 1.0
    
    # Pętla IPF
    for iteration in range(max_iter):
        
        # A_i = 1 / sum_j (B_j * D_j * f_cij)
        df['B_D_f'] = df['B_j'] * df['D_j'] * df['f_cij']
        sum_A = df.groupby('origin_id')['B_D_f'].transform('sum')
        new_A_i = np.where(sum_A > 0, 1.0 / sum_A, 0)
        # NORMALIZE A
        new_A_i = new_A_i / new_A_i.mean() if new_A_i.mean() > 0 else new_A_i
        
        diff_A = np.abs(new_A_i - df['A_i']).max()
        df['A_i'] = new_A_i
        
        # B_j = 1 / sum_i (A_i * O_i * f_cij)
        df['A_O_f'] = df['A_i'] * df['O_i'] * df['f_cij']
        sum_B = df.groupby('dest_id')['A_O_f'].transform('sum')
        new_B_j = np.where(sum_B > 0, 1.0 / sum_B, 0)
        # NORMALIZE B
        new_B_j = new_B_j / new_B_j.mean() if new_B_j.mean() > 0 else new_B_j
        
        diff_B = np.abs(new_B_j - df['B_j']).max()
        df['B_j'] = new_B_j
        
        if max(diff_A, diff_B) < tol:
            print(f"  ✓ IPF zbiegł po {iteration+1} iteracjach (Błąd = {max(diff_A, diff_B):.6e})")
            break
    else:
        print(f"  ! UWAGA: Osiągnięto max_iter ({max_iter}) przy błędzie {max(diff_A, diff_B):.6e}")

    # 3. Przepływy
    df['T_ij'] = df['A_i'] * df['O_i'] * df['B_j'] * df['D_j'] * df['f_cij']
    
    # Aktualizacja D_jobs by odpowiadało skalowanym wartościom dla dalszych etapów
    df[D_col] = df['D_j']
    
    df.drop(columns=['B_D_f', 'A_O_f', 'O_i', 'D_j'], inplace=True, errors='ignore')
    return df


In [3]:
def run_city_gravity_pipeline(city, apply_mock_warsaw=False):
    """
    Przeprowadza ciąg analityczny grawitacji na dane miasto w wariancie trójmodelowym (Metoda Porównawcza z etapu metodologii):
    - Model A: Sztywny Niski (beta = 0.03) - "Captive Rider"
    - Model B: Sztywny Wysoki (beta = 0.15) - "Choice/Oportunistyczny"
    - Model C (dynamiczne beta)
    """
    print(f"\n{'='*50}\nUruchamianie silnika Grawitacyjnego w podejściu 3-wariantowym: {city.upper()}\n{'='*50}")
    
    out_dir = OUTPUTS_E4 / city
    (out_dir / "accessibility").mkdir(parents=True, exist_ok=True)
    (out_dir / "calibration").mkdir(parents=True, exist_ok=True)
    
    wtc_path = INPUTS_E3 / city / "wtc" / "wtc_matrix.parquet"
    units_path = INPUTS_E3 / city / "od" / "od_units.csv"
    origins_geo_path = INPUTS_E3 / city / "od" / "origins.geojson"
    
    if not wtc_path.exists() or not units_path.exists() or not origins_geo_path.exists():
        print(f"  ✗ Brak plików podstawowych Etapu III dla {city}. Omijam.")
        return
        
    print(f"  Ładowanie macierzy podróży WTC z Parquet... ({wtc_path.name})")
    df_wtc = pd.read_parquet(wtc_path)
    
    if 'cost_min_strict' in df_wtc.columns:
        cost_col = 'cost_min_strict'
    else:
        cost_col = 'cost_min'
        
    df_wtc = df_wtc.dropna(subset=[cost_col]).copy()
    df_wtc.rename(columns={cost_col: 'cost'}, inplace=True)
    
    print(f"  Załadowano połączeń o sensownym WTC: {len(df_wtc)}")
    
    df_od = pd.read_csv(units_path)
    origins_geo = gpd.read_file(origins_geo_path)
    
    # Ustalanie wskaźnika zamożności do Modelu C
    wealth_col = 'income_median' if city == 'paris' else 'wealth_index'
    if wealth_col not in origins_geo.columns:
        print(f"  ! UWAGA: Brak kolumny wskaźnika zamożności '{wealth_col}' w origins.geojson. Dynamiczne beta będzie równe gęstości lub zablokowane.")
        origins_geo['beta_dynamic'] = 0.08
    else:
        # Przydzielenie kwintyli (0 do 4) na odfiltrowanym rozkładzie
        valid_idx = origins_geo[wealth_col].notna()
        if valid_idx.sum() > 0:
            try:
                quantiles = pd.qcut(origins_geo.loc[valid_idx, wealth_col], 5, labels=False, duplicates='drop')
                # Przeskaluj kwintyle na beta między 0.03 a 0.15
                # np. 5 przedziałów (0, 1, 2, 3, 4) -> (0.03, 0.06, 0.09, 0.12, 0.15)
                # Maksymalny indeks to przeważnie 4, ale może być mniej przy duplikatach, więc ustalamy max bezpiecznie.
                max_q = quantiles.max() if quantiles.max() > 0 else 1
                beta_mapped = 0.03 + (0.15 - 0.03) * (quantiles / max_q)
                origins_geo.loc[valid_idx, 'beta_dynamic'] = beta_mapped
            except ValueError:
                # Fallback w przypadku problemów z dystrybucją (wszystkie równe)
                origins_geo.loc[valid_idx, 'beta_dynamic'] = 0.08
        
        # Braki danych imputujemy uśrednioną betą
        origins_geo['beta_dynamic'] = origins_geo['beta_dynamic'].fillna(0.08)
    
    df_origins_geo_sub = origins_geo[['origin_id', 'beta_dynamic']]
    
    if 'D_jobs' not in df_od.columns:
        df_od['D_jobs'] = np.nan
    if 'O_pop' not in df_od.columns:
        df_od['O_pop'] = np.nan
        
    origins = df_od[['unit_id', 'O_pop']].rename(columns={'unit_id': 'origin_id'})
    destinations = df_od[['unit_id', 'D_jobs']].rename(columns={'unit_id': 'dest_id'})
    
    if city == 'warsaw' and apply_mock_warsaw:
        print(f"  [METODOLOGIA] Warszawa: Użycie zastępczych danych o pracy z przygotowanego skryptu.")
        dummy_grid_path = OUTPUTS / "etap1" / "warsaw" / "spatial" / "grid_1km_metric_dummy_jobs.geojson"
        if dummy_grid_path.exists():
            gdf_dummy = gpd.read_file(dummy_grid_path)
            dest_geo_path = INPUTS_E3 / city / "od" / "destinations.geojson"
            if dest_geo_path.exists():
                dest_geo = gpd.read_file(dest_geo_path)
                joined = gpd.sjoin(dest_geo, gdf_dummy[['employment', 'geometry']], how='left', predicate='within')
                joined_agg = joined.groupby('dest_id')['employment'].sum().reset_index()
                destinations = destinations.merge(joined_agg, on='dest_id', how='left')
                destinations['D_jobs'] = destinations['employment']
                destinations.drop(columns=['employment'], inplace=True)
                print("  Mapowanie zapasowego zatrudnienia (employment) udane.")
            else:
                print("  ! Brak destinations.geojson do mapowania !")
        else:
            print("  ! Błąd braku skryptu zapasowego warszawy !")

    if city == 'warsaw' and destinations['D_jobs'].isna().all():
        destinations['D_jobs'] = destinations['D_jobs'].fillna(np.random.randint(100, 2000, size=len(destinations)))

    merged = df_wtc.merge(origins, on='origin_id', how='inner')
    merged = merged.merge(destinations, on='dest_id', how='inner')
    
    # Dodanie dynamicznej bety
    merged = merged.merge(df_origins_geo_sub, on='origin_id', how='left')
    
    merged['O_pop'] = merged['O_pop'].fillna(0)
    merged['D_jobs'] = merged['D_jobs'].fillna(0)
    
    merged = merged[(merged['O_pop'] > 0) & (merged['D_jobs'] > 0) & (merged['cost'] > 0)].copy()
    
    if len(merged) == 0:
        print("  ✗ Pusta macierz złączeniowa! Brak pasujących pop względem job.")
        return
        
    models = {
        'A': {'name': 'Model A (beta = 0.03)', 'beta': 0.03},
        'B': {'name': 'Model B (beta = 0.15)', 'beta': 0.15},
        'C': {'name': 'Model C (dynamiczne beta)', 'beta': 'beta_dynamic'}
    }
    
    access_gdf = origins_geo.copy()
    
    for m_key, m_info in models.items():
        print(f"  Faza modelowania grawitacyjnego {m_info['name']}...")
        results = doubly_constrained_gravity(merged, 'O_pop', 'D_jobs', beta=m_info['beta'])
        
        # Oszacowanie Pasywnego Indeksu Hansena do mapowania Gini
        df_grouped = results.groupby('origin_id')
        access_df = df_grouped.apply(lambda x: (x['D_jobs'] * x['f_cij']).sum(), include_groups=False).reset_index(name=f'accessibility_index_{m_key}')
        
        access_gdf = access_gdf.merge(access_df, on='origin_id', how='left')
        
        flows_file = out_dir / "accessibility" / f"flows_{m_key}.parquet"
        results.to_parquet(flows_file, index=False)
        
    # Ewaluacja Komparatystyczna (Wskaźnik Względnej Różnicy RDI)
    access_gdf['RDI_C_vs_A'] = (access_gdf['accessibility_index_C'] - access_gdf['accessibility_index_A']) / access_gdf['accessibility_index_A'].replace(0, np.nan)
    access_gdf['RDI_C_vs_B'] = (access_gdf['accessibility_index_C'] - access_gdf['accessibility_index_B']) / access_gdf['accessibility_index_B'].replace(0, np.nan)

    access_geo_file = out_dir / "accessibility" / "accessibility_grid_3models.geojson"
    access_gdf.to_file(access_geo_file, driver="GeoJSON")
    
    calib = {
        "city": city,
        "models": models,
        "n_origins": int(access_gdf.shape[0]),
        "n_flows": int(merged.shape[0]),
        "matrix_used": cost_col,
        "wealth_metric_used": wealth_col,
        "calculation_time": str(datetime.utcnow())
    }
    with open(out_dir / "calibration" / "beta_params_3models.json", 'w') as f:
        json.dump(calib, f, indent=2)
        
    print(f"  ✓ Utworzono 3 warianty modeli dla {city.upper()}. Wyeksportowano warstwy Parquet & Geojson.")


In [4]:
# Przeprowadzenie silnika grawitacji na każdą metropolię w nowych standardach 3-wariantowych
for c in CITIES:
    is_waw = (c == 'warsaw')
    run_city_gravity_pipeline(c, apply_mock_warsaw=is_waw)



Uruchamianie silnika Grawitacyjnego w podejściu 3-wariantowym: PARIS
  Ładowanie macierzy podróży WTC z Parquet... (wtc_matrix.parquet)
  Załadowano połączeń o sensownym WTC: 274413
  Faza modelowania grawitacyjnego Model A (Sztywny Niski beta=0.03)...
  ! UWAGA: Osiągnięto max_iter (200) przy błędzie 7.509997e+02
  Faza modelowania grawitacyjnego Model B (Sztywny Wysoki beta=0.15)...
  ! UWAGA: Osiągnięto max_iter (200) przy błędzie 1.945840e+03
  Faza modelowania grawitacyjnego Model C (Dynamiczny na zamożności)...
  ! UWAGA: Osiągnięto max_iter (200) przy błędzie 1.105692e+03
  ✓ Utworzono 3 warianty modeli dla PARIS. Wyeksportowano warstwy Parquet & Geojson.

Uruchamianie silnika Grawitacyjnego w podejściu 3-wariantowym: DUBLIN
  Ładowanie macierzy podróży WTC z Parquet... (wtc_matrix.parquet)
  Załadowano połączeń o sensownym WTC: 189219
  Faza modelowania grawitacyjnego Model A (Sztywny Niski beta=0.03)...
  ! UWAGA: Osiągnięto max_iter (200) przy błędzie 3.289876e-02
  Faza mod